In [1]:
pip install setfit sentence-transformers scikit-learn pandas datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from scipy.linalg import sqrtm
from setfit import SetFitModel, SetFitTrainer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# --- Load Data ---
print("--- Loading Data ---")
source_df = pd.read_csv('IMDB_Dataset_Cleaned.csv')
target_df = pd.read_csv('SST2_Dataset_Cleaned.csv')

# Assuming your CSVs are structured as described
source_texts = source_df['review'].tolist()
source_labels = source_df['label'].tolist()

target_texts_full = target_df['review'].tolist()
target_labels_full = target_df['label'].tolist()

# Create a small target domain training set (few-shot)
target_df_train, target_df_val = train_test_split(target_df, test_size=0.8, stratify=target_df['label'], random_state=42)
target_texts_train = target_df_train['review'].tolist()
target_labels_train = target_df_train['label'].tolist()
target_texts_val = target_df_val['review'].tolist()
target_labels_val = target_df_val['label'].tolist()

print(f"Source data size: {len(source_texts)}")
print(f"Target train data size (few-shot): {len(target_texts_train)}")
print(f"Target validation data size: {len(target_texts_val)}")
print(f"Target full data size: {len(target_texts_full)}")
print("-" * 30)

# --- 1. Use Pre-trained Sentence Transformers for Feature Extraction ---
print("--- Step 1: Feature Extraction ---")
transformer_models = {
    "all-MiniLM-L6-v2": "all-MiniLM-L6-v2",
    "distilroberta-base": "distilroberta-base",
    "bert-base-uncased": "bert-base-uncased",
}
embeddings = {}

for name, model_name in transformer_models.items():
    print(f"Loading model: {model_name}")
    model = SentenceTransformer(model_name)
    embeddings[name] = {
        "source": model.encode(source_texts),
        "target_train": model.encode(target_texts_train),
        "target_val": model.encode(target_texts_val),
        "target_full": model.encode(target_texts_full)
    }
    print(f"Embeddings for {name} extracted.")
print("-" * 30)

# --- 2. Apply a Lightweight Domain Adaptation Technique (CORAL) ---
print("--- Step 2: Domain Adaptation (CORAL) ---")

def coral(source, target):
    """
    Applies CORAL domain adaptation.
    """
    source_mean = np.mean(source, axis=0)
    target_mean = np.mean(target, axis=0)
    source_cov = np.cov(source.T) + np.eye(source.shape[1]) * 1e-6  # Add small identity for numerical stability
    target_cov = np.cov(target.T) + np.eye(target.shape[1]) * 1e-6

    Cs_inv_sqrt = sqrtm(np.linalg.inv(source_cov))
    Ct_sqrt = sqrtm(target_cov)
    A = Cs_inv_sqrt @ Ct_sqrt
    source_corrected = (source - source_mean) @ A + target_mean
    return source_corrected

coral_embeddings = {}
for name, emb in embeddings.items():
    print(f"Applying CORAL for {name}")
    source_corrected = coral(emb["source"], emb["target_train"])  # Using target_train for alignment
    coral_embeddings[name] = {
        "source": source_corrected,
        "target_train": emb["target_train"],
        "target_val": emb["target_val"],
        "target_full": emb["target_full"]
    }
print("-" * 30)

# --- 3. Use SetFit for Few-Shot Classification ---
print("--- Step 3: Few-Shot Classification (SetFit) ---")

# Encode labels if they are not already numerical (they should be 0 or 1)
source_labels_encoded = np.array(source_labels)
target_labels_train_encoded = np.array(target_labels_train)
target_labels_val_encoded = np.array(target_labels_val)
target_labels_full_encoded = np.array(target_labels_full)

setfit_results = {}

for name, emb in embeddings.items():
    print(f"Training SetFit model with {name} embeddings (without DA)")
    model = SetFitModel(train_embeddings=emb["source"], train_labels=source_labels_encoded)
    trainer = SetFitTrainer(model=model, train_dataset=(emb["source"], source_labels_encoded))
    trainer.train()
    y_pred = model.predict(emb["target_val"])
    accuracy = accuracy_score(target_labels_val_encoded, y_pred)
    f1 = f1_score(target_labels_val_encoded, y_pred, average='weighted')
    setfit_results[f"{name}_no_da"] = {"accuracy": accuracy, "f1": f1}
    print(f"{name} (No DA) - Accuracy: {accuracy:.4f}, F1: {f1:.4f}")

for name, emb in coral_embeddings.items():
    print(f"Training SetFit model with {name} embeddings (with CORAL)")
    model = SetFitModel(train_embeddings=emb["source"], train_labels=source_labels_encoded)
    trainer = SetFitTrainer(model=model, train_dataset=(emb["source"], source_labels_encoded))
    trainer.train()
    y_pred = model.predict(emb["target_val"])
    accuracy = accuracy_score(target_labels_val_encoded, y_pred)
    f1 = f1_score(target_labels_val_encoded, y_pred, average='weighted')
    setfit_results[f"{name}_coral"] = {"accuracy": accuracy, "f1": f1}
    print(f"{name} (CORAL) - Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
print("-" * 30)

# --- 4. Benchmark Architectures ---
print("--- Step 4: Benchmarking ---")
print("SetFit Results:")
for model_name, metrics in setfit_results.items():
    print(f"{model_name}: Accuracy = {metrics['accuracy']:.4f}, F1 = {metrics['f1']:.4f}")

# Train and evaluate on the full target dataset for comparison
full_data_results = {}
for name, model_name in transformer_models.items():
    print(f"\nTraining Logistic Regression on full target data with {name} embeddings")
    model = SentenceTransformer(model_name)
    target_embeddings_full = model.encode(target_texts_full)
    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(target_embeddings_full, target_labels_full_encoded)
    target_embeddings_val = embeddings[name]["target_val"]
    y_pred_full = clf.predict(target_embeddings_val)
    accuracy_full = accuracy_score(target_labels_val_encoded, y_pred_full)
    f1_full = f1_score(target_labels_val_encoded, y_pred_full, average='weighted')
    full_data_results[f"{name}_full_data"] = {"accuracy": accuracy_full, "f1": f1_full}
    print(f"{name} (Full Data) - Accuracy: {accuracy_full:.4f}, F1: {f1_full:.4f}")

print("\nFull Data Training Results:")
for model_name, metrics in full_data_results.items():
    print(f"{model_name}: Accuracy = {metrics['accuracy']:.4f}, F1 = {metrics['f1']:.4f}")

print("-" * 30)
print("Implementation Plan Completed!")

--- Loading Data ---


C:\Users\ivans\anaconda3\Lib\site-packages\sklearn\utils\validation.py:605: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
C:\Users\ivans\anaconda3\Lib\site-packages\sklearn\utils\validation.py:614: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):


Source data size: 50000
Target train data size (few-shot): 13644
Target validation data size: 54577
Target full data size: 68221
------------------------------
--- Step 1: Feature Extraction ---
Loading model: all-MiniLM-L6-v2


C:\Users\ivans\anaconda3\Lib\site-packages\ipywidgets\widgets\widget.py:438: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


KeyboardInterrupt

